In [ ]:
import cdsapi

dataset = "sis-tourism-fire-danger-indicators"
request = {
    "time_aggregation": "seasonal_indicators",
    "product_type": "multi_model_mean_case",
    "variable": ["seasonal_fire_weather_index"],
    "experiment": "rcp4_5",
    "period": [
        "2016_2020",
        "2021_2025"
    ],
    "version": "v2_0"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

d1 = xr.open_dataset('/work/wildfirerisk/mean-model_rcp45_fwi-mean-jjas_20160101_20201231_v2.nc')['fwi-mean-jjas']
d2 = xr.open_dataset('/work/wildfirerisk/mean-model_rcp45_fwi-mean-jjas_20210101_20251231_v2.nc')['fwi-mean-jjas']
d = xr.concat((d1,d2), 'time')
dlat, dlon = d.lat.values.ravel(), d.lon.values.ravel()
d = d.values.reshape(d.shape[0], -1)
mask = ~np.isnan(d).all(0)
d, dlat, dlon = d[:,mask], dlat[mask], dlon[mask]

def get_fwi_at_point(lat, lon, time_index=0):
    dist = np.sqrt((dlat - lat)**2 + (dlon - lon)**2)
    min_idx = dist.argmin()
    return float(d[time_index, min_idx])

In [180]:
import pandas as pd
import json

template = pd.read_json("/work/wildfirerisk/vlm_cot_predictions.json").T
predictions, answers = {}, {}
for lat_lon, row in template.iterrows():
    img_path = row.img_path
    if 'europe' not in img_path:
        continue
    lat, lon = map(float, lat_lon.split('_'))
    if 'no_fire' in img_path:
        fwi = np.mean([get_fwi_at_point(lat,lon,i) for i in range(d.shape[0])])
    else:
        year = int(img_path.split('/')[-2])-2015
        year = min(year, 9)
        fwi = get_fwi_at_point(lat,lon,year)
    answers[lat_lon] =  {'answer': min(float(fwi)/10, 9), 'ground_truth': row.ground_truth, 'img_path': img_path}
    predictions[lat_lon] = {'answer': int(min(np.floor(fwi/10), 9)), 'ground_truth': row.ground_truth, 'img_path': img_path}

with open("/work/wildfirerisk/fwi-mean-jjas_predictions.json", 'w') as f:
    json.dump(predictions, f)

with open("/work/wildfirerisk/fwi-answers_predictions.json", 'w') as f:
    json.dump(answers, f)

In [12]:
import pandas as pd
import json
from sklearn.preprocessing import quantile_transform

template = pd.read_json("/work/wildfirerisk/vlm_cot_predictions.json").T
fwis = {}
for lat_lon, row in template.iterrows():
    img_path = row.img_path
    if 'europe' not in img_path:
        continue
    lat, lon = map(float, lat_lon.split('_'))
    if 'no_fire' in img_path:
        fwi = np.mean([get_fwi_at_point(lat,lon,i) for i in range(d.shape[0])])
    else:
        year = int(img_path.split('/')[-2])-2015
        year = min(year, 9)
        fwi = get_fwi_at_point(lat,lon,year)
    fwis[lat_lon] = fwi

fwis = pd.Series(fwis)
fwis = pd.Series(quantile_transform(fwis.values.reshape(-1,1))[:,0], index=fwis.index).to_dict()

predictions, answers = {}, {}
for lat_lon, row in template.iterrows():
    if 'europe' not in row.img_path:
        continue
    fwi = fwis[lat_lon]
    answers[lat_lon] =  {'answer': min(float(fwi*10), 9), 'ground_truth': row.ground_truth, 'img_path': row.img_path}
    predictions[lat_lon] = {'answer': int(min(np.floor(fwi*10), 9)), 'ground_truth': row.ground_truth, 'img_path': row.img_path}

with open("/work/wildfirerisk/fwi-mean-jjas_predictions_normed.json", 'w') as f:
    json.dump(predictions, f)

with open("/work/wildfirerisk/fwi-answers_predictions_normed.json", 'w') as f:
    json.dump(answers, f)

In [14]:
template

,model_response,answer,ground_truth,img_path,raster_path
42.0349_20.1082,system\nYou are a helpful assistant.\nuser\nYo...,7,3,/work/wildfirerisk/europe_dataset/fire_images/...,/work/wildfirerisk/europe_dataset/fire_masks/2...
43.9522_16.703,system\nYou are a helpful assistant.\nuser\nYo...,6,1,/work/wildfirerisk/europe_dataset/fire_images/...,/work/wildfirerisk/europe_dataset/fire_masks/2...
42.6779_18.1789,system\nYou are a helpful assistant.\nuser\nYo...,8,1,/work/wildfirerisk/europe_dataset/fire_images/...,/work/wildfirerisk/europe_dataset/fire_masks/2...
52.7888_7.3928,system\nYou are a helpful assistant.\nuser\nYo...,4,2,/work/wildfirerisk/europe_dataset/fire_images/...,/work/wildfirerisk/europe_dataset/fire_masks/2...
38.0229_22.6303,system\nYou are a helpful assistant.\nuser\nYo...,5,1,/work/wildfirerisk/europe_dataset/fire_images/...,/work/wildfirerisk/europe_dataset/fire_masks/2...
...,...,...,...,...,...
42.0108_-110.8262,system\nYou are a helpful assistant.\nuser\nYo...,7,6,/work/wildfirerisk/large_dataset/satellite_ima...,/work/wildfirerisk/large_dataset/normalised_ri...
41.7256_-110.8908,system\nYou are a helpful assistant.\nuser\nYo...,7,5,/work/wildfirerisk/large_dataset/satellite_ima...,/work/wildfirerisk/large_dataset/normalised_ri...
41.8159_-110.9102,system\nYou are a helpful assistant.\nuser\nYo...,8,5,/work/wildfirerisk/large_dataset/satellite_ima...,/work/wildfirerisk/large_dataset/normalised_ri...
41.9062_-110.9296,system\nYou are a helpful assistant.\nuser\nYo...,6,5,/work/wildfirerisk/large_dataset/satellite_ima...,/work/wildfirerisk/large_dataset/normalised_ri...


In [15]:
pd.DataFrame(predictions).T

,answer,ground_truth,img_path
42.0349_20.1082,2,3,/work/wildfirerisk/europe_dataset/fire_images/...
43.9522_16.703,2,1,/work/wildfirerisk/europe_dataset/fire_images/...
42.6779_18.1789,5,1,/work/wildfirerisk/europe_dataset/fire_images/...
52.7888_7.3928,0,2,/work/wildfirerisk/europe_dataset/fire_images/...
38.0229_22.6303,8,1,/work/wildfirerisk/europe_dataset/fire_images/...
...,...,...,...
60.8135_92.9116,0,0,/work/wildfirerisk/europe_dataset/no_fire_imag...
74.6752_94.0103,0,0,/work/wildfirerisk/europe_dataset/no_fire_imag...
75.7003_95.3996,0,0,/work/wildfirerisk/europe_dataset/no_fire_imag...
69.5657_96.7131,0,0,/work/wildfirerisk/europe_dataset/no_fire_imag...
